# A town is a graph: which streets carry the traffic?

Companion to [Week 06 §11](../README.md#11-graphs-in-architecture-three-applications)
and `Lecture_06/graph_applications/street_network.py`. Street corners are
nodes, street segments are edges. The question moves from one route to
**all routes at once**: if everybody travels the shortest way, which
streets does the town depend on?

This notebook also hands the graph to **networkx**, and walks into the trap
that waits there.

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in Lecture/notebooks/ or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

import networkx as nx
from compas.datastructures import Graph

## Step 1 — a town with a river
A 10 by 8 grid of corners. A river runs between rows 3 and 4, and the
north–south streets cross it only where there is a bridge.

In [ ]:
COLS, ROWS, RIVER = 10, 8, 3


def town(bridges=(2, 7)):
    graph = Graph()
    for j in range(ROWS):
        for i in range(COLS):
            graph.add_node((i, j), x=float(i), y=j + (0.6 if j > RIVER else 0.0), z=0.0)
    for j in range(ROWS):
        for i in range(COLS):
            if i + 1 < COLS:
                graph.add_edge((i, j), (i + 1, j), kind="street")
            if j + 1 < ROWS and (j != RIVER or i in bridges):
                graph.add_edge((i, j), (i, j + 1), kind="bridge" if j == RIVER else "street")
    return graph


def draw_town(graph, title, colours=None, size=(560, 420)):
    return draw_graph(graph, title, node_labels=False, edge_colors=colours, size=size,
                      node_colors={n: "#adb5bd" for n in graph.nodes()})


base = town()
bridges = list(base.edges_where(kind="bridge"))
print(base.number_of_nodes(), "corners,", base.number_of_edges(), "street segments, bridges:", bridges)
draw_town(base, "two bridges (orange)", {edge: "#e8590c" for edge in bridges})

## Step 2 — the trap: every street is one-way
COMPAS stores an edge in the direction you added it (Week 06 §2), so
`to_networkx()` returns a **directed** graph. Here every street was added
west → east or south → north. Ask networkx how far you can get from the
north-east corner.

In [ ]:
directed = base.to_networkx()
print(type(directed).__name__)
corner = (COLS - 1, ROWS - 1)
print("reachable from the north-east corner, as converted:", len(nx.descendants(directed, corner)), "corners")

streets = directed.to_undirected()
print("after .to_undirected():                            ", len(nx.descendants(streets, corner)), "corners")

Zero. Nobody in the north-east corner can go anywhere, and nothing raised
an error. A route-finding function on the directed graph would either fail
with *"not strongly connected"* or, worse, return quietly wrong numbers.
**`.to_undirected()` is the fix; one line, easy to forget.**

## Step 3 — one trip, then all of them
Give every segment its length, then ask for one shortest route between two
corners on opposite banks. It has to use a bridge.

In [ ]:
def as_streets(graph):
    G = graph.to_networkx().to_undirected()
    for u, v in G.edges():
        G.edges[u, v]["length"] = graph.edge_length((u, v) if graph.has_edge((u, v)) else (v, u))
    return G


streets = as_streets(base)
route = nx.shortest_path(streets, (0, 0), (5, 7), weight="length")
print("from (0, 0) to (5, 7):", len(route) - 1, "segments,",
      round(nx.path_weight(streets, route, weight="length"), 1), "blocks long")
draw_graph(base, "one trip across the river", node_labels=False, path=route, size=(560, 420),
           node_colors={n: "#adb5bd" for n in base.nodes()})

There are 80 corners, so 3160 pairs. **Edge betweenness** is the share of
all those shortest trips that pass along a segment. Count it for one bridge
by hand first, so the library call below is not magic.

In [ ]:
west = bridges[0]
pairs = using = 0
corners = list(streets.nodes())
for a_index, a in enumerate(corners):
    for b in corners[a_index + 1:]:
        trip = nx.shortest_path(streets, a, b, weight="length")
        pairs += 1
        using += any({u, v} == set(west) for u, v in zip(trip, trip[1:]))
print(f"{using} of {pairs} trips use the west bridge: {using / pairs:.3f}")

In [ ]:
def betweenness(graph):
    scores = nx.edge_betweenness_centrality(as_streets(graph), weight="length", normalized=True)
    return {(u, v) if graph.has_edge((u, v)) else (v, u): score for (u, v), score in scores.items()}


score = betweenness(base)
print(f"networkx says {score[west]:.3f} for the west bridge")
print("(a small difference is expected: where several routes tie, networkx shares the")
print(" trip between them, and the loop above followed one route per trip)")
top = max(score.values())
for edge in sorted(score, key=score.get, reverse=True)[:4]:
    print(f"  {edge}  {base.edge_attribute(edge, 'kind'):7s} {score[edge]:.2f}")
draw_town(base, "betweenness: orange carries the most trips", {e: ramp(s / top) for e, s in score.items()})

The two bridges, and the streets leading to them, carry the town. Nothing
in the drawing of a regular grid says so; the count does.

## Step 4 — close a bridge
Roadworks on the west bridge. Two numbers: the average trip length, and how
much of the town's traffic the busiest segment now takes.

In [ ]:
def report(name, graph):
    scores = betweenness(graph)
    busiest = max(scores, key=scores.get)
    trip = nx.average_shortest_path_length(as_streets(graph), weight="length")
    print(f"{name:22s} average trip {trip:5.2f} blocks   busiest segment {busiest} takes {scores[busiest]:.2f}")
    return scores


closed = town(bridges=(7,))
report("both bridges open", base)
closed_score = report("west bridge closed", closed)
top = max(closed_score.values())
row(draw_town(base, "both open", {e: ramp(s / max(score.values())) for e, s in score.items()}, size=(450, 360)),
    draw_town(closed, "west bridge closed", {e: ramp(s / top) for e, s in closed_score.items()}, size=(450, 360)))

## Step 5 — where should a third bridge go?
A design question with a measurable answer: try every free position,
rebuild the graph, measure. Eight graphs, a second or two.

In [ ]:
results = {}
for column in range(COLS):
    if column in (2, 7):
        continue
    candidate = town(bridges=(2, 7, column))
    scores = betweenness(candidate)
    results[column] = (nx.average_shortest_path_length(as_streets(candidate), weight="length"),
                       max(scores.values()))
    print(f"third bridge at column {column}: average trip {results[column][0]:.3f}, "
          f"busiest segment {results[column][1]:.3f}")
best = min(results, key=lambda column: results[column][0])
print("shortest average trip with the bridge at column", best)

In this regular town both numbers point to the middle (columns 4 and 5 tie,
by symmetry), and a bridge at either end changes almost nothing. In a less
regular town the two criteria need not agree: the bridge that shortens trips
most is not always the one that best relieves the busiest street. Deciding
between them is the designer's job; producing the table is the graph's.

## Your turn
1. Remove `.to_undirected()` from `as_streets` and run Step 3. What exactly
   goes wrong, and would you have noticed without Step 2?
2. Make the east bridge a slow one: multiply its `length` by 3 in
   `as_streets`. How does the traffic redistribute?
3. Which **corner** is the most central? Use
   `nx.betweenness_centrality` (nodes instead of edges) and draw it with
   `node_colors`.
4. Replace the regular grid by a real one: delete ten random street
   segments (keep the town connected: check with `nx.is_connected`) and
   look at the betweenness again.